In [17]:
import socket
import time
import xml.etree.ElementTree as ET

class GilsonSession:
    def __init__(self, ip, admin_port=50185):
        self.ip = ip
        self.admin_port = admin_port
        self.session_port = None
        self.session_socket = None
        self._connect()

    def _connect(self):
        message = """
        <Message MessageId="1984" BlockCount="1" BlockId="0">
        <Gilson>
            <GilsonConnect>Gilson Ethernet Utility</GilsonConnect>
        </Gilson>
        </Message>
        """.replace('\n', '\r\n').strip()

        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.settimeout(5)
            s.connect((self.ip, self.admin_port))
            s.sendall(message.encode('ascii'))
            response = s.recv(4096).decode('ascii')

        if "<ParameterName>ResponseValue</ParameterName>" in response:
            port = int(response.split("<ParameterValue>")[2].split("</ParameterValue>")[0])
            self.session_port = port

            # open persistent socket
            self.session_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            self.session_socket.settimeout(5)
            self.session_socket.connect((self.ip, self.session_port))
        else:
            raise RuntimeError("Failed to retrieve assigned session port from Gilson.")

    def send_raw_command(self, xml_payload):
        try:
            self.session_socket.sendall(xml_payload.encode('ascii'))
            time.sleep(1)
            buffer = b""
            while True:
                try:
                    part = self.session_socket.recv(8192)
                    if not part:
                        break
                    buffer += part
                    if b"</Message>" in part:
                        break
                except socket.timeout:
                    break
            response = buffer.decode('ascii', errors='ignore')
            print(f"Full XML response:\n{response}\n")
            return response
        except Exception as e:
            print(f"Error sending raw command:", e)
            return None

    def make_command(self, command_name, device="GX-27x", device_id=35, sequence_number=1, command_type="Local", synchronize=True, selected=False):
        message_id = int(time.time()) % 10000
        xml = f"""
<Message MessageId="{message_id}" BlockCount="2" BlockId="0">
<Gilson>
    <CommandData>
        <Commands>
            <Command>
                <CommandType>{command_type}</CommandType>
                <CommandInfo>
                    <InstrumentInfo>
                        <DeviceName>{device}</DeviceName>
                        <DeviceId>{device_id}</DeviceId>
                    </InstrumentInfo>
                    <SequenceNumber>{sequence_number}</SequenceNumber>
                    <Synchronize>{str(synchronize)}</Synchronize>
                    <Selected>{str(selected)}</Selected>
                    <CommandXML>
                        <CommandName>{command_name}</CommandName>
                    </CommandXML>
                </CommandInfo>
            </Command>
        </Commands>
    </CommandData>
</Gilson>
</Message>
<Message MessageId="{message_id}" BlockCount="2" BlockId="1">
</Message>
""".replace('\n', '\r\n').strip()
        return xml

    def send_command(self, command_name, device="GX-27x", device_id=35, sequence_number=1):
        xml = self.make_command(command_name, device, device_id, sequence_number, command_type="Local")
        return self.send_raw_command(xml)

    def send_immediate_command(self, command_name, device="GX-27x", device_id=35, sequence_number=1):
        xml = self.make_command(command_name, device, device_id, sequence_number, command_type="Immediate")
        return self.send_raw_command(xml)

    def close(self):
        if self.session_socket:
            self.session_socket.close()
            self.session_socket = None


In [18]:
# Example usage:
# gilson = GilsonSession("192.168.1.101")
# gilson.send_command("Admin", device="System", device_id=-1, sequence_number=0)
# gilson.send_command("Identify")

In [19]:
g = GilsonSession("192.168.1.101")

In [20]:
g.send_immediate_command("Admin", device="System", device_id=-1, sequence_number=0)

Full XML response:
<Message MessageId="2162" BlockCount="2" BlockId="0"><Gilson><Status><SystemInformation><IPAddress>192.168.1.101</IPAddress><SubnetMask>255.255.255.0</SubnetMask><DefaultGateway>192.168.1.1</DefaultGateway><MAC>00:40:9d:a2:56:92</MAC><LeaseInformation><LeaseActive>False</LeaseActive></LeaseInformation></SystemInformation><Devices><Device><DeviceName>GX-27x</DeviceName><DeviceId>35</DeviceId><SerialNumber>261J7N253</SerialNumber><Connected>True</Connected><Commands><Command><CommandName>Get Input Contacts</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Parameter><ParameterName>ResponseValue</ParameterName><ParameterValue><ReturnParameters><ReturnParameter><ParameterName>Contact A</ParameterName><ParameterValue>Off</ParameterValue></ReturnParameter><ReturnParameter><ParameterName>Contact B</ParameterName><ParameterValue>Off</ParameterValue></ReturnParameter><ReturnParameter><ParameterName>Saf

'<Message MessageId="2162" BlockCount="2" BlockId="0"><Gilson><Status><SystemInformation><IPAddress>192.168.1.101</IPAddress><SubnetMask>255.255.255.0</SubnetMask><DefaultGateway>192.168.1.1</DefaultGateway><MAC>00:40:9d:a2:56:92</MAC><LeaseInformation><LeaseActive>False</LeaseActive></LeaseInformation></SystemInformation><Devices><Device><DeviceName>GX-27x</DeviceName><DeviceId>35</DeviceId><SerialNumber>261J7N253</SerialNumber><Connected>True</Connected><Commands><Command><CommandName>Get Input Contacts</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Parameter><ParameterName>ResponseValue</ParameterName><ParameterValue><ReturnParameters><ReturnParameter><ParameterName>Contact A</ParameterName><ParameterValue>Off</ParameterValue></ReturnParameter><ReturnParameter><ParameterName>Contact B</ParameterName><ParameterValue>Off</ParameterValue></ReturnParameter><ReturnParameter><ParameterName>Safety</ParameterName

In [5]:
g.send_immediate_command("Identify")

Full XML response:
<Message MessageId="2148" BlockCount="2" BlockId="0"><Gilson><Status><SystemInformation><IPAddress>192.168.1.101</IPAddress><SubnetMask>255.255.255.0</SubnetMask><DefaultGateway>192.168.1.1</DefaultGateway><MAC>00:40:9d:a2:56:92</MAC><LeaseInformation><LeaseActive>False</LeaseActive></LeaseInformation></SystemInformation><Devices><Device><DeviceName>GX-27x</DeviceName><DeviceId>35</DeviceId><SerialNumber>261J7N253</SerialNumber><Connected>True</Connected><Commands><Command><CommandName>Get Input Contacts</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Parameter><ParameterName>ResponseValue</ParameterName><ParameterValue><ReturnParameters><ReturnParameter><ParameterName>Contact A</ParameterName><ParameterValue>Off</ParameterValue></ReturnParameter><ReturnParameter><ParameterName>Contact B</ParameterName><ParameterValue>Off</ParameterValue></ReturnParameter><ReturnParameter><ParameterName>Saf

'<Message MessageId="2148" BlockCount="2" BlockId="0"><Gilson><Status><SystemInformation><IPAddress>192.168.1.101</IPAddress><SubnetMask>255.255.255.0</SubnetMask><DefaultGateway>192.168.1.1</DefaultGateway><MAC>00:40:9d:a2:56:92</MAC><LeaseInformation><LeaseActive>False</LeaseActive></LeaseInformation></SystemInformation><Devices><Device><DeviceName>GX-27x</DeviceName><DeviceId>35</DeviceId><SerialNumber>261J7N253</SerialNumber><Connected>True</Connected><Commands><Command><CommandName>Get Input Contacts</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Parameter><ParameterName>ResponseValue</ParameterName><ParameterValue><ReturnParameters><ReturnParameter><ParameterName>Contact A</ParameterName><ParameterValue>Off</ParameterValue></ReturnParameter><ReturnParameter><ParameterName>Contact B</ParameterName><ParameterValue>Off</ParameterValue></ReturnParameter><ReturnParameter><ParameterName>Safety</ParameterName

In [8]:
g.send_command("Identify", device="GX-27x", device_id=35)

Full XML response:




''